# Giao thức v2 — đánh giá cuối trên Google Colab (GPU)

Notebook chạy **đánh giá cuối** của giao thức v2 (`audit/PREREG_v2.md`) trên mẫu kiểm định B — mở tập kiểm thử của B
**đúng một lần** — rồi kiểm định, xuất bảng/hình/macro sang báo cáo:

| Bước | Script | Ra |
|---|---|---|
| Dữ liệu: mẫu A, mẫu B, đặc trưng | `00_sample_hm.py`, `00_sample_hm.py --holdout`, `21_build_features.py` | `data/processed/hm/` (lưu lên Drive cho lần sau) |
| Kiểm tra: unit test v2, md5 hai mẫu, khoá kiểm thử | `tests/test_v2*.py`, `23_final_v2.py` (hàm `lock_errors`) | báo **SẴN SÀNG** / danh sách lỗi |
| Đánh giá cuối: 5 seed + ablation NeuMF-F (seed 42) | `23_final_v2.py` | `outputs/v2/final/seed*/` |
| Kiểm định, bảng, hình, macro LaTeX | `24_report_v2.py`, `19_check_report.py` | `outputs/v2/final/*.csv`, `Report DACNTT/content/tables/v2/`, `media/figures/v2/` |

**Điều kiện trước khi chạy** (làm trên máy local): tinh chỉnh đã xong (`audit/v2/best_configs.json` đủ 13 mô hình),
mọi code + `audit/PREREG_v2.md` + `audit/v2/` đã **commit và push**. Khoá kiểm thử so mã băm mã nguồn với lúc tinh chỉnh,
nên không sửa `src/`, `scripts/v2_common.py`, `scripts/22_tune_v2.py`, `configs/v2.yaml` sau khi tinh chỉnh.

Thời gian ước lượng trên GPU T4: dữ liệu 15–25 phút (lần đầu, tải Kaggle), đánh giá cuối 2–4 giờ. Giữ tab mở.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi -L || echo "KHÔNG có GPU — vào Runtime > Change runtime type > GPU"

## 2. Cấu hình và mount Google Drive

Drive giữ dữ liệu đã xử lý cho các lần chạy sau, **bản sao nhật ký truy cập tập kiểm thử** (phòng Colab ngắt giữa chừng)
và kết quả. Repo private: thêm secret `GITHUB_TOKEN` (biểu tượng 🔑), bật quyền truy cập cho notebook.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/sulinh3625/DU-AN-CNTT.git"
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/neumf_colab"            # dữ liệu + kết quả
REASON = "Đánh giá cuối v2 theo PREREG_v2 — Colab GPU"       # ghi vào audit/test_access_log.csv
SEEDS = "42 2024 2025 2026 3407"
MIRROR_LOG = f"{DRIVE_DIR}/v2_test_access_log.csv"           # bản sao nhật ký truy cập test trên Drive

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
CLONE_URL = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
print("Drive:", DRIVE_DIR, "| token:", "có" if token else "không (repo phải public)")

## 3. Clone repo và cài thư viện

In [ ]:
%cd /content
!rm -rf DU-AN-CNTT
!git clone -q --depth 1 --branch $BRANCH $CLONE_URL DU-AN-CNTT
%cd /content/DU-AN-CNTT/neumf_project
!git log --oneline -1
!pip install -q -r requirements.txt

> Nếu pip báo đã đổi phiên bản `numpy`/`pandas`: `Runtime → Restart session`, chạy lại ô 2 và `%cd /content/DU-AN-CNTT/neumf_project`.

## 4. Dữ liệu: mẫu A, mẫu B và đặc trưng

Ô dưới dùng bản trên Drive nếu có (`DRIVE_DIR/v2_data/`); nếu chưa, tải `transactions_train.csv`, `articles.csv`,
`customers.csv` từ Kaggle (secret `KAGGLE_USERNAME` + `KAGGLE_KEY`, hoặc `DRIVE_DIR/kaggle.json`), lấy mẫu, dựng đặc trưng
rồi lưu lên Drive. Cuối ô so mã MD5 của hai mẫu với `audit/PREREG_v2.md` — lệch thì **dừng**.

In [ ]:
import hashlib, os, shutil, subprocess
from pathlib import Path

PROC, RAW, CACHE = Path("data/processed/hm"), Path("data/raw/hm"), Path(DRIVE_DIR) / "v2_data"
PROC.mkdir(parents=True, exist_ok=True); RAW.mkdir(parents=True, exist_ok=True)
FILES = {PROC / "hm500k_transactions.csv": CACHE / "hm500k_transactions.csv",
         PROC / "hm500k_b_transactions.csv": CACHE / "hm500k_b_transactions.csv",
         PROC / "catalog_v2.npz": CACHE / "catalog_v2.npz",
         PROC / "daily_sales_v2.npz": CACHE / "daily_sales_v2.npz",
         RAW / "customers.csv": CACHE / "customers.csv"}

if all(c.exists() for c in FILES.values()):
    for dst, src in FILES.items():
        shutil.copy(src, dst)
    print("Dùng dữ liệu đã xử lý trên Drive:", CACHE)
else:
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception:
        kj = Path(DRIVE_DIR) / "kaggle.json"
        if not kj.exists():
            raise SystemExit("Thiếu Kaggle: thêm secret KAGGLE_USERNAME/KAGGLE_KEY hoặc đặt kaggle.json vào " + DRIVE_DIR)
        Path("~/.kaggle").expanduser().mkdir(exist_ok=True)
        shutil.copy(kj, Path("~/.kaggle/kaggle.json").expanduser())
        os.chmod(Path("~/.kaggle/kaggle.json").expanduser(), 0o600)
    for f in ("transactions_train.csv", "articles.csv", "customers.csv"):
        subprocess.run(["kaggle", "competitions", "download", "-c", "h-and-m-personalized-fashion-recommendations",
                        "-f", f, "-p", str(RAW)], check=True)
        z = RAW / f"{f}.zip"
        if z.exists():
            subprocess.run(["unzip", "-o", "-q", str(z), "-d", str(RAW)], check=True); z.unlink()
    subprocess.run(["python", "scripts/00_sample_hm.py"], check=True)
    subprocess.run(["python", "scripts/00_sample_hm.py", "--holdout"], check=True)
    subprocess.run(["python", "scripts/21_build_features.py"], check=True)
    CACHE.mkdir(parents=True, exist_ok=True)
    for dst, src in FILES.items():
        shutil.copy(dst, src)
    print("Đã lưu dữ liệu đã xử lý lên Drive:", CACHE)

def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

want = {"hm500k_transactions.csv": "13f48afe5133d79bcb9c0832c4ee287d",
        "hm500k_b_transactions.csv": "19ba444b60694b8240b1df41d834cd98"}
for name, h in want.items():
    got = md5(PROC / name)
    print(name, got, "✓" if got == h else "✗ KHÁC audit/PREREG_v2.md — DỪNG, kiểm tra dữ liệu")

## 5. Kiểm tra trước khi chạy (không chấm test)

Unit test của giao thức v2, tóm tắt cấu hình đã tinh chỉnh, rồi kiểm tra đủ điều kiện **khoá kiểm thử** bằng chính hàm
`lock_errors` của `23_final_v2.py`. Báo lỗi thì **dừng**, sửa trên máy local, commit + push, chạy lại notebook từ ô 3.

Nếu một lần chạy trước bị ngắt giữa chừng, ô này chép các dòng nhật ký truy cập test từ bản sao trên Drive vào
`audit/test_access_log.csv`, để khoá kiểm thử biết seed nào đã mở tập kiểm thử.

In [ ]:
!python -m pytest -q -p no:cacheprovider tests/test_v2.py tests/test_v2_final.py 2>&1 | tail -3

import csv, importlib.util, json, sys
from types import SimpleNamespace
from pathlib import Path

log, mirror = Path("audit/test_access_log.csv"), Path(MIRROR_LOG)
if mirror.exists():
    have = log.read_text(encoding="utf-8").splitlines() if log.exists() else []
    new = [l for l in mirror.read_text(encoding="utf-8").splitlines()[1:] if l and l not in have]
    if new:
        with log.open("a", encoding="utf-8", newline="") as f:
            f.write("\n".join(new) + "\n")
        print(f"Đã chép {len(new)} dòng nhật ký từ Drive (lần chạy trước bị ngắt):", *new, sep="\n  ")

best = json.loads(Path("audit/v2/best_configs.json").read_text(encoding="utf-8"))
for m, b in best.items():
    print(f"{m:11s} val NDCG@10 {b['val']['NDCG@10']:.5f}  {b['params']}")

sys.path.insert(0, "scripts")
spec = importlib.util.spec_from_file_location("final_v2", "scripts/23_final_v2.py")
final = importlib.util.module_from_spec(spec); spec.loader.exec_module(final)
args = SimpleNamespace(reason=REASON, seeds=[int(s) for s in SEEDS.split()], allow_rerun=False, allow_code_change=False,
                       max_epochs=final.V.CFG["training"]["max_epochs"], models=None, skip_ablation=False)
errs = final.lock_errors(args, final.V.provenance(), best)
print("\nSẴN SÀNG — có thể chạy mục 6." if not errs else "\nCHƯA SẴN SÀNG:\n  - " + "\n  - ".join(errs))

## 6. Đánh giá cuối trên tập kiểm thử của mẫu B (mở đúng một lần)

Mỗi seed ghi một dòng `audit/test_access_log.csv` (và bản sao `MIRROR_LOG` trên Drive) **trước** khi chấm. Log chạy ghi
song song lên Drive. Nếu Colab ngắt giữa chừng: chạy lại từ ô 2 → ô 5 (chép nhật ký từ Drive), rồi chạy ô này với
`SEEDS` chỉ gồm các seed **chưa** có dòng nhật ký; seed đang chạy dở đã mở tập kiểm thử → muốn chạy lại phải thêm
`--allow-rerun` và ghi vào `audit/PREREG_v2.md` mục 10 (lệch kế hoạch).

In [ ]:
LOG = f"{DRIVE_DIR}/final_v2.log"
!python -u scripts/23_final_v2.py --reason "$REASON" --seeds $SEEDS --mirror-log "$MIRROR_LOG" 2>&1 | tee -a "$LOG"
done = [s for s in SEEDS.split() if Path(f"outputs/v2/final/seed{s}/results.json").exists()]
print("Seed đã xong:", done)
if len(done) != len(SEEDS.split()):  # tee nuốt mã lỗi -> tự kiểm tra đầu ra
    raise SystemExit(f"23_final_v2.py chưa xong mọi seed — xem {LOG}")

## 7. Kiểm định, bảng, hình, macro LaTeX và đối chiếu câu chữ

In [ ]:
!python scripts/24_report_v2.py 2>&1 | tee "$DRIVE_DIR/report_v2.log"
!python scripts/19_check_report.py 2>&1 | tee "$DRIVE_DIR/check_report_v2.log"

## 8. Lưu kết quả lên Drive và tải về máy

In [ ]:
import shutil, time
from pathlib import Path
from google.colab import files

stamp = time.strftime("%Y%m%d-%H%M")
dest = Path(DRIVE_DIR) / f"results_v2_{stamp}"
shutil.copytree("outputs/v2/final", dest / "v2_final", dirs_exist_ok=True)   # gồm checkpoint seed 42 (*.pt, bpr.npz)
shutil.copy("audit/test_access_log.csv", dest / "test_access_log.csv")
report = Path("../Report DACNTT")
shutil.copytree(report / "content" / "tables" / "v2", dest / "report" / "content" / "tables" / "v2", dirs_exist_ok=True)
shutil.copytree(report / "media" / "figures" / "v2", dest / "report" / "media" / "figures" / "v2", dirs_exist_ok=True)
zip_path = shutil.make_archive(f"/content/results_v2_{stamp}", "zip", dest)
print("Đã lưu:", dest)
files.download(zip_path)

### Chép về repo trên máy (sau khi giải nén)

| Trong zip | Chép vào repo |
|---|---|
| `v2_final/` | `neumf_project/outputs/v2/final/` |
| `test_access_log.csv` | `neumf_project/audit/test_access_log.csv` (ghi đè — đã gồm dòng cũ + dòng mới) |
| `report/` | `Report DACNTT/` (ghi đè `content/tables/v2/`, `media/figures/v2/`) |

Rồi trên máy: `python scripts/19_check_report.py`, biên dịch báo cáo (`Report DACNTT/compile.bat`), viết phần thảo luận
theo kết quả, và commit. Mạng nơ-ron trên GPU có thể lệch nhẹ ở chữ số cuối so với CPU — ghi rõ máy đã chạy (macro
`\VData{run}{device}` tự sinh từ `results.json`).

## 9. Xem nhanh kết quả

In [ ]:
import pandas as pd
from IPython.display import display, Image
summary = pd.read_csv("outputs/v2/final/summary.csv")
display(summary[["name", "NDCG@10_mean", "NDCG@10_std", "NDCG@10_old_mean", "NDCG@10_new_mean", "oldonly_NDCG@10_mean",
                 "val_NDCG@10"]].round(5))
sig = pd.read_csv("outputs/v2/final/significance.csv")
display(sig[["id", "A", "B", "mean_A", "mean_B", "rel_diff", "ci_low", "ci_high", "p_holm", "verdict"]].round(5))
for f in sorted(Path("outputs/v2/final/figures").glob("*.png")):
    print(f.name); display(Image(str(f), width=900))